In [ ]:
from google.colab import drive 
drive.mount('/content/drive')

In [ ]:
import pdb
import os
import numpy as np

ComputeInterArrival = True
DescretesizeLength = False
DirectionLengthCombined = True
NormalizeLength = True
NormalizeInterArrival = True
MaxLength = 1434
MaxInterArrival = 1
Starting_point = 0
StartingPointMultiply = 13
Num_of_extracted_subflow = 100
PaddingEnable = True
PadAtTheBegining = True
PaddingThreshold = 20


CompureStatisticsInThisScript = True

NumOfCrossValidationFolds = 1

np.random.seed(10)

timestep = 120
SkipPacketsForSampling = 1
IncrementalSampling = False
NumberOfSamplesUntiIncrement = 10000
IncrementalStepMultiplier = 1


def loadData(dirPath, class_label,  extractedFlows = 0):
    #If it is not set, use the global value
    if extractedFlows == 0:
        extractedFlows = Num_of_extracted_subflow

    pathDir = %cd /content/drive/MyDrive/QUIC\ Dataset

    datalist = []
    labellist = []
    FileCounter = 0
    FlowCounter = 0.0000009
    SubflowCounter = 0

    # added by Shahbaz
    custom_features = [
         0,    #timestamp
        1,  # RelativeTime
        2,  # length
         3    #Direction
    ]

    for folder, subs, files in os.walk(dirPath):

        # if FlowCounter > 1000:
        #     break

        np.random.shuffle(files)

        for file in files:
            filename = folder + "/" + file
            with open(filename) as f:
                
                FileCounter += 1
                
                EntireFile = []
                for line in f:
                    data = line.split()
                    try:
                        EntireFile.append(data)
                    except:
                        print(EntireFile)
                        pdb.set_trace()
                try:
                    EntireFile = np.array(EntireFile).astype(np.float)
                except:
                    print(EntireFile)
                    pdb.set_trace()

                FileLenght = len(EntireFile)

                if CompureStatisticsInThisScript:
                    Duration = EntireFile[-1,1]
                    Bandwidth = np.sum(EntireFile[:,2])/Duration
                    temp_label = [Bandwidth, Duration, class_label]

                SubflowFromAFile = 0

                #Skip the fist few packets in the file
                if(Starting_point!=0):
                    for jjj in range(Starting_point):
                        line = f.readline()
                        if not line:
                            break

                for subflow in range(extractedFlows):

                    startingPoint = 0
                    if subflow == 0:
                        startingPoint = 0
                    else:
                        startingPoint = np.random.randint(1, FileLenght-timestep)
                    # startingPoint = Starting_point + subflow*StartingPointMultiply
                    
                    linedata = []
                    Prev_time = 0;  #Time of the first packet in the subflow
    
                    numOfSamples = 0
                    i = startingPoint
                    SkipSamples = SkipPacketsForSampling
                    while(numOfSamples < timestep):
                        if i>=FileLenght:
                            break

                        data = list(EntireFile[i])  #To clone the list, not refering to the same list
                        
                        #shahbaz: To descretesize the the length
                        if DescretesizeLength:
                            data[2] = str(int(int(data[2])/100))

                        if DirectionLengthCombined:
                            if data[3]=="0":
                                if float(data[2])>0:
                                    data[2] = str(-1 * float(data[2]))
                                    
                        if NormalizeLength:
                            data[2] = str(float(data[2])/MaxLength)
                            
                        if ComputeInterArrival:
                            if i==startingPoint:
                                Prev_time = float(data[1])
                                data[1] = str(0)
                            else:
                                temporary = str(float(data[1]) - Prev_time)
                                Prev_time = float(data[1])
                                data[1] = temporary
                        if NormalizeInterArrival:
                            ttt = float(data[1]) / MaxInterArrival
                            if ttt > 1:
                                ttt=1
                            data[1]=(ttt-0.5)*2
                            

                        try:
                            data2 = [float(data[j]) for j in custom_features]
                        except (IndexError, ValueError) as e:
                            pass
                            print("Couldn't retrieve all data",filename)
                        else:
                            linedata += data2
                
                        numOfSamples += 1
                        i += SkipSamples
                        if IncrementalSampling:
                            if numOfSamples % NumberOfSamplesUntiIncrement == 0:
                                SkipSamples = int(SkipSamples*IncrementalStepMultiplier)

    
                    if (len(linedata) < len(custom_features) * timestep):
                        if (PaddingThreshold > len(linedata)/len(custom_features) ):
                            continue
                        #print(linedata)
                        if (PaddingEnable):
                            while(len(linedata) < len(custom_features) * timestep):
                                pad = []
                                pad.extend(np.ones(len(custom_features)) * 0)
                                if PadAtTheBegining:
                                    pad.extend(linedata)
                                    linedata = pad
                                else:
                                    linedata.extend(pad)
                            #print(linedata)
                        else:
                            continue
                    np.nan_to_num(linedata)
                    datalist.append(linedata)

                    SubflowCounter+=1
                    SubflowFromAFile+=1

                total_labels = [temp_label] * SubflowFromAFile
                labellist.extend(total_labels)
                FlowCounter+=1
                print(filename,temp_label)
    ratio = SubflowCounter/FlowCounter
    print(dirPath + ":" + str(FlowCounter) + "/" + str(pathDir) + " - Subflows:" + str(SubflowCounter) + " - Ratio:", str(ratio))
    return (np.array(datalist), np.array(labellist))


if __name__ == "__main__":

    BaseDirectory = '%cd /content/drive/MyDrive/QUIC Dataset/pretraining'
    (data1, label1) = loadData(BaseDirectory + "/Google Drive", 1, extractedFlows=1)
    (data2, label2) = loadData(BaseDirectory + "/Youtube", 2,  extractedFlows=1)
    (data3, label3) = loadData(BaseDirectory + "/Google Doc", 3,  extractedFlows=1)
    (data4, label4) = loadData(BaseDirectory + "/Google Search", 4, extractedFlows=1)
    (data5, label5) = loadData(BaseDirectory + "/Google Music", 5, extractedFlows=1)

    test_size = 30
    val_size = 30
    train1 = data1[:-(test_size+val_size)]
    train2 = data2[:-(test_size+val_size)]
    train3 = data3[:-(test_size+val_size)]
    train4 = data4[:-(test_size+val_size)]
    train5 = data5[:-(test_size+val_size)]
    val1 = data1[-(test_size+val_size):-test_size]
    val2 = data2[-(test_size+val_size):-test_size]
    val3 = data3[-(test_size+val_size):-test_size]
    val4 = data4[-(test_size+val_size):-test_size]
    val5 = data5[-(test_size+val_size):-test_size]
    test1 = data1[-test_size:]
    test2 = data2[-test_size:]
    test3 = data3[-test_size:]
    test4 = data4[-test_size:]
    test5 = data5[-test_size:]

    trainL1 = label1[:-(test_size+val_size)]
    trainL2 = label2[:-(test_size+val_size)]
    trainL3 = label3[:-(test_size+val_size)]
    trainL4 = label4[:-(test_size+val_size)]
    trainL5 = label5[:-(test_size+val_size)]
    valL1 = label1[-(test_size+val_size):-test_size]
    valL2 = label2[-(test_size+val_size):-test_size]
    valL3 = label3[-(test_size+val_size):-test_size]
    valL4 = label4[-(test_size+val_size):-test_size]
    valL5 = label5[-(test_size+val_size):-test_size]
    testL1 = label1[-test_size:]
    testL2 = label2[-test_size:]
    testL3 = label3[-test_size:]
    testL4 = label4[-test_size:]
    testL5 = label5[-test_size:]

    train_data = np.concatenate((train1, train2, train3, train4, train5), axis=0)
    val_data = np.concatenate((val1, val2, val3, val4, val5), axis=0)
    test_data = np.concatenate((test1, test2, test3, test4, test5), axis=0)

    train_label = np.concatenate((trainL1, trainL2, trainL3, trainL4, trainL5), axis=0)
    val_label = np.concatenate((valL1, valL2, valL3, valL4, valL5), axis=0)
    test_label = np.concatenate((testL1, testL2, testL3, testL4, testL5), axis=0)

    np.save("trainData.npy", train_data)
    np.save("trainLabel.npy", train_label)

    np.save("valData.npy", val_data)
    np.save("valLabel.npy", val_label)

    np.save("testData.npy", test_data)
    np.save("testLabel.npy", test_label)

    print(train_data.shape, train_label.shape)
    print(val_data.shape, val_label.shape)
    print(test_data.shape, test_label.shape)

In [ ]:
%cd /content/drive/MyDrive/QUIC Dataset/pretraining/ pretraining/ 
%ls

In [ ]:
import numpy as np
from keras.models import Model
from keras.layers import Dense
from keras.layers import multiply
from keras.layers import Flatten
from keras.layers import Input
from keras.layers.convolutional import Conv1D, MaxPooling1D
from keras.layers import Activation
from keras.optimizers import Adam

timestep = 60
np.random.seed(10)

num_class = 5
train_sample_per_class = 20
lambda_value = 1


trainData = np.load("trainData.npy")
trainlabel = np.load("trainLabel.npy")
trainData = trainData[:timestep*2]
trainlabel = trainlabel[:timestep*2]
trainlabel = trainlabel.astype(int)

trainmask = np.zeros((trainlabel.shape[0],256))

class_counter = np.zeros((num_class))
train_size = trainlabel.shape[0]
j = 0
for i in range(train_size):
    class_id = trainlabel[i,2] - 1
    if class_counter[class_id] < train_sample_per_class:
        trainmask[i, :] = 1
        j += 1
        class_counter[class_id] += 1
print("unmasked samples: ", str(np.sum(trainmask==1)/256))


valData = np.load("valData.npy")
valLabel = np.load("valLabel.npy")
valData = valData[:timestep*2]
valLabel = valLabel[:timestep*2]

valLabel = valLabel.astype(int)
valmask = np.ones((valLabel.shape[0], 256))
valmask[:,:]=1


testData = np.load("testData.npy")
testLabel = np.load("testLabel.npy")
testData = testData[:timestep*2]
testLabel = testLabel[:timestep*2]

testLabel = testLabel.astype(int)
testmask = np.ones((testLabel.shape[0], 256))
testmask[:,:]=1

for i in range(trainlabel.shape[0]):
    #Categorizing Bandwidth
    if trainlabel[i, 0] < 10000:
        trainlabel[i, 0] = 1
    elif trainlabel[i, 0] < 50000:
        trainlabel[i, 0] = 2
    elif trainlabel[i, 0] < 100000:
        trainlabel[i, 0] = 3
    elif trainlabel[i, 0] < 1000000:
        trainlabel[i, 0] = 4
    else:
        trainlabel[i, 0] = 5
    #Categorizing Duration
    if trainlabel[i, 1] < 10:
        trainlabel[i, 1] = 1
    elif trainlabel[i, 1] < 30:
        trainlabel[i, 1] = 2
    elif trainlabel[i, 1] < 60:
        trainlabel[i, 1] = 3
    else:
        trainlabel[i, 1] = 4

for i in range(valLabel.shape[0]):
    #Categorizing Bandwidth
    if valLabel[i, 0] < 10000:
        valLabel[i, 0] = 1
    elif valLabel[i, 0] < 50000:
        valLabel[i, 0] = 2
    elif valLabel[i, 0] < 100000:
        valLabel[i, 0] = 3
    elif valLabel[i, 0] < 1000000:
        valLabel[i, 0] = 4
    else:
        valLabel[i, 0] = 5
    #Categorizing Duration
    if valLabel[i, 1] < 10:
        valLabel[i, 1] = 1
    elif valLabel[i, 1] < 30:
        valLabel[i, 1] = 2
    elif valLabel[i, 1] < 60:
        valLabel[i, 1] = 3
    else:
        valLabel[i, 1] = 4


for i in range(testLabel.shape[0]):
    #Categorizing Bandwidth
    if testLabel[i, 0] < 10000:
        testLabel[i, 0] = 1
    elif testLabel[i, 0] < 50000:
        testLabel[i, 0] = 2
    elif testLabel[i, 0] < 100000:
        testLabel[i, 0] = 3
    elif testLabel[i, 0] < 1000000:
        testLabel[i, 0] = 4
    else:
        testLabel[i, 0] = 5
    #Categorizing Duration
    if testLabel[i, 1] < 10:
        testLabel[i, 1] = 1
    elif testLabel[i, 1] < 30:
        testLabel[i, 1] = 2
    elif testLabel[i, 1] < 60:
        testLabel[i, 1] = 3
    else:
        testLabel[i, 1] = 4


train_size = trainlabel.shape[0]
Y_train1 = np.zeros((train_size,5))
a = np.arange(train_size,trainlabel[:,0],-1)
Y_train1[np.arange([train_size,trainlabel[:,0],-1])] = 1
Y_train2 = np.zeros((train_size,4))
Y_train2[np.arange(train_size),trainlabel[:,1]-1] = 1
Y_train3 = np.zeros((train_size,5))
Y_train3[np.arange(train_size),trainlabel[:,2]-1] = 1

val_size = valLabel.shape[0]
Y_val1 = np.zeros((val_size,5))
Y_val1[np.arange(val_size),valLabel[:,0]-1] = 1
Y_val2 = np.zeros((val_size,4))
Y_val2[np.arange(val_size),valLabel[:,1]-1] = 1
Y_val3 = np.zeros((val_size,5))
Y_val3[np.arange(val_size),valLabel[:,2]-1] = 1

test_size = testLabel.shape[0]
Y_test1 = np.zeros((test_size,5))
Y_test1[np.arange(test_size),testLabel[:,0]-1] = 1
Y_test2 = np.zeros((test_size,4))
Y_test2[np.arange(test_size),testLabel[:,1]-1] = 1
Y_test3 = np.zeros((test_size,5))
Y_test3[np.arange(test_size),testLabel[:,2]-1] = 1


trainData = trainData.reshape((trainData.shape[0], timestep, 2))
testData = testData.reshape((testData.shape[0], timestep, 2))
valData = valData.reshape((valData.shape[0], timestep, 2))

def base_model():

    model_input = Input(shape=(timestep,2))
    mask_input = Input(shape=(256,))

    x = Conv1D(32, 3, activation='relu')(model_input)
    x = Conv1D(32, 3, activation='relu')(x)
    x = MaxPooling1D(pool_size=(2))(x)

    x = Conv1D(64, 3, activation='relu')(x)
    x = Conv1D(64, 3, activation='relu')(x)
    x = MaxPooling1D(pool_size=(2))(x)

    x = Conv1D(128, 3, activation='relu')(x)
    x = Conv1D(128, 3, activation='relu')(x)
    x = MaxPooling1D(pool_size=(2))(x)

    x = Flatten()(x)

    x = Dense(256)(x)
    x = Activation('relu')(x)
    x = Dense(256)(x)
    x = Activation('relu')(x)

    output1 = Dense(5, activation='softmax', name='Bandwidth')(x)

    output2 = Dense(4, activation='softmax', name='Duration')(x)

    x3 = multiply([x,mask_input])
    output3 = Dense(5, activation='softmax', name='Class')(x3)

    model = Model(inputs=[model_input,mask_input], outputs=[output1, output2, output3])
    opt = Adam(clipnorm = 1.)
    model.compile(loss=['categorical_crossentropy', 'categorical_crossentropy', 'categorical_crossentropy'], loss_weights=[1,1,lambda_value], optimizer=opt, metrics=['accuracy'])

    return model

model = base_model()

model.fit([trainData,trainmask], [Y_train1, Y_train2, Y_train3],
          validation_data = ([valData, valmask], [Y_val1, Y_val2, Y_val3]),
          batch_size = 64, epochs = 20, verbose = True, shuffle = True)

result = model.evaluate([testData, testmask], [Y_test1, Y_test2, Y_test3])
print(result)